In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# 1. Point directly to your certificate folder
cert_path = "/Workspace/Users/rishikeshmate09@gmail.com/Health care ML"

with open(f"{cert_path}/ca.pem", "r") as f: ca_string = f.read()
with open(f"{cert_path}/service.cert", "r") as f: cert_string = f.read()
with open(f"{cert_path}/service.key", "r") as f: key_string = f.read()

# 2. Connect to Aiven (Batch Read)
kafka_bootstrap_servers = "kafka-114b7e84-rishikeshmate09-5283.l.aivencloud.com:21915"
raw_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", "patient-vitals") \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.truststore.certificates", ca_string) \
    .option("kafka.ssl.keystore.certificate.chain", cert_string) \
    .option("kafka.ssl.keystore.key", key_string) \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

# 3. Define the Blueprint (Schema) for your JSON
# (Update these field names if your Colab generator uses different ones!)
json_schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("blood_pressure", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

# 4. The Magic: Unpack the JSON and expand it into columns
silver_df = raw_df.select(
    from_json(col("value").cast("string"), json_schema).alias("parsed_data")
).select("parsed_data.*")

# 5. Display the clean data!
display(silver_df)